# Gold: export MosaicML shards

Streams a published `gold.manifest_rows` release's images out of their source archives and writes them into per-split MosaicML shard sets (`streaming.MDSWriter`), for both local-machine and Databricks ML-cluster training to read via `streaming.StreamingDataset`.

Shards are a derived, fully rebuildable cache, never a second source of truth for image bytes -- see `docs/decisions/006-stream-archives-no-blob-storage.md`. No image bytes are stored anywhere in this project outside the original source archives and, temporarily, these shard files -- nothing is ever extracted to a Volume as an individual image file. Rerunning this notebook for the same `dataset_version` always fully rebuilds every split's shard directory (`MDSWriter`'s `exist_ok=True` clears each shard directory before writing) -- a shard set can always be regenerated identically from the manifest and archives, so a full rebuild is a safe, simple default rather than needing incremental-sync logic.

**Retention is an open question, not decided yet** -- shards accumulate under `GOLD_ROOT` until someone manually deletes them; there is no default cleanup step and no cloud-storage lifecycle policy assumed. See `docs/decisions/009-gold-shard-retention-undecided.md`. `data_platform.spark_io.remove_expired_exports` exists and is available whenever/if a retention decision gets made -- it's just not called from this notebook by default.

Configured by `config/gold/exports/<EXPORT_NAME>.yaml`, mirroring `config/gold/manifests/<name>.yaml`'s convention. Run `30_create_gold_manifest.ipynb` for the target `dataset_version` before this notebook -- this one only reads `gold.manifest_rows`/`gold.manifest_registry`, it never writes to them. Requires `mosaicml-streaming==0.13.0` (pinned in `pyproject.toml`) -- declared as a dependency on this notebook's serverless Environment (not installed via a `%pip install` cell), so it's available without a per-run install.

In [ ]:
%run ./_setup_env


In [ ]:
from pathlib import Path
import tempfile

from data_platform.files import stage_archives_and_extract_metadata
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths
from data_platform.shard_export import write_gold_shards_for_splits
from data_platform.spark_io import (
    load_manifest_rows_for_export,
    print_export_outputs,
)

GOLD_TABLES = {"manifest_rows": "gold.manifest_rows"}

EXPORT_NAME = "sample-v1"
EXPORT_CONFIG = load_yaml_config(CONFIG_ROOT / "gold" / "exports" / f"{EXPORT_NAME}.yaml")

DATASET_VERSION = EXPORT_CONFIG["dataset_version"]
SHARD_SIZE_LIMIT_BYTES = EXPORT_CONFIG["shard_size_limit_bytes"]

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
GOLD_ROOT = f"{STORAGE_CONFIG['storage_root'].rstrip('/')}/gold"
EXPORT_SHARD_ROOT = f"{GOLD_ROOT}/{DATASET_VERSION}/shards"

print(f"Exporting dataset_version={DATASET_VERSION!r} to {EXPORT_SHARD_ROOT}")


## Discover Release Datasets

Look up which `dataset_key`s and `preprocessing_version` the target `dataset_version` actually contains via `gold.manifest_registry` -- fails fast if `30_create_gold_manifest.ipynb` hasn't been run yet for this release, or if it spans more than one `preprocessing_version` (not designed for a mixed-preprocessing release).

In [ ]:
registry_rows = spark.sql(
    f"SELECT * FROM gold.manifest_registry WHERE dataset_version = '{DATASET_VERSION}'"
).collect()
if not registry_rows:
    raise ValueError(
        f"No gold.manifest_registry row for dataset_version={DATASET_VERSION!r} -- "
        "run 30_create_gold_manifest.ipynb for this release first."
    )
registry_row = registry_rows[0]

DATASET_KEYS = registry_row["dataset_keys"]
preprocessing_versions = registry_row["preprocessing_versions"]
if len(preprocessing_versions) != 1:
    raise ValueError(
        f"dataset_version={DATASET_VERSION!r} spans preprocessing_versions={preprocessing_versions!r} "
        "-- shard export isn't designed for a mixed-preprocessing release."
    )
PREPROCESSING_VERSION = preprocessing_versions[0]

print(f"dataset_keys: {DATASET_KEYS}")
print(f"preprocessing_version: {PREPROCESSING_VERSION}")


## Stage Archives

Every referenced dataset's source archives need a local staged copy before their images can be streamed into shards -- the same `stage_archives_and_extract_metadata` helper Bronze/Silver already use, called once per `dataset_key` in this release.

In [ ]:
staged_path_by_archive_uri = {}

for dataset_key in DATASET_KEYS:
    dataset_config = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / f"{dataset_key}.yaml")
    dataset_config["landing_root"] = STORAGE_CONFIG["landing_root"]
    dataset_config["storage_root"] = STORAGE_CONFIG["storage_root"]
    dataset_layout = build_layout(dataset_config)
    archives = resolve_archive_paths(
        dataset_config["archives"], dataset_layout["landing_paths"], dataset_layout["bronze_paths"]
    )
    local_stage_root = Path(tempfile.gettempdir()) / dataset_key / "archive_stage"
    stage_archives_and_extract_metadata(archives, local_stage_root)
    for archive in archives:
        staged_path_by_archive_uri[archive["archive_dbfs_path"]] = archive["staged_archive_path"]

print(f"Staged {len(staged_path_by_archive_uri)} archive(s) across {len(DATASET_KEYS)} dataset(s)")


## Load Manifest Rows

`load_manifest_rows_for_export` driver-collects `gold.manifest_rows` for this release, grouped by `split` -- metadata only (`image_id`, `dataset_key`, `bronze_uri`, `source_checksum`, `label`, `group_id`), no image bytes touched yet. `source_checksum` is carried through so `write_gold_shards_for_splits` can verify each image is still what the manifest recorded (see `docs/decisions/008-immutable-source-archives-checksum-verified.md`).

In [ ]:
rows_by_split = load_manifest_rows_for_export(spark, GOLD_TABLES, DATASET_VERSION)

for split, rows in rows_by_split.items():
    print(f"{split}: {len(rows)} manifest rows")


## Write Shards

`write_gold_shards_for_splits` streams every split's sampled images out of their staged source archives in one pass and writes them into per-split `streaming.MDSWriter` shard sets, sized by `SHARD_SIZE_LIMIT_BYTES` -- a source archive commonly contributes images to more than one split, so this opens every split's writer up front and streams each archive exactly once rather than once per split. Raises if any manifest row can't be found in its source archive, or if a found image's checksum no longer matches what the manifest recorded -- either way means the source archive changed after this manifest was published, breaking its reproducibility guarantee, not a routine per-row rejection (see `docs/decisions/008-immutable-source-archives-checksum-verified.md`).

In [ ]:
shard_dir_by_split = {split: f"{EXPORT_SHARD_ROOT}/{split}" for split in rows_by_split}

per_split_summaries = write_gold_shards_for_splits(
    rows_by_split, staged_path_by_archive_uri, shard_dir_by_split, SHARD_SIZE_LIMIT_BYTES
)

for split, summary in per_split_summaries.items():
    print(f"{split}: wrote {summary['sample_count']} samples to {summary['shard_dir']}")


## Review Outputs

`print_export_outputs` cross-checks each split's written sample count against `gold.manifest_registry`'s counts for this `dataset_version` -- raises on any mismatch, since a shard set should always contain exactly the manifest's rows.

In [ ]:
print_export_outputs(spark, display, DATASET_VERSION, per_split_summaries)
